# GLADA: Incremental Dataset Evaluation

Evaluates **one dataset at a time** so you can check them off as they become available.
Run each section independently. Results merge into the same `results.json`.

**Priority order** (datasets already on Drive first):
1. ASVspoof 2019 LA eval — already done, verify
2. ITW — already done, verify
3. ASVspoof 2021 LA — keys on Drive, just needs extraction
4. ASVspoof 2021 DF — audio + keys on Drive
5. WaveFake — archive on Drive, needs LJSpeech bonafide
6. CodecFake+ — shared drive, needs extraction

## Setup

In [1]:
# === Cell 1: Setup (always run first) ===
import os, sys, json, time, datetime, tarfile, zipfile
from pathlib import Path
from dataclasses import asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9  # total_memory not total_mem
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    if 'a100' in gpu_name.lower() or 'h100' in gpu_name.lower():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

AMP_DTYPE = torch.bfloat16 if device.type == 'cuda' and (
    'a100' in torch.cuda.get_device_name(0).lower() or
    'h100' in torch.cuda.get_device_name(0).lower()
) else torch.float16

RESULTS_DIR = '/content/drive/MyDrive/Jakob/Experiments/comprehensive_eval'
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(RESULTS_DIR, 'results.json')

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} existing model results.')
else:
    all_results = {}
    print('No existing results found.')

# Package
GLADA_PKG_PATH = '/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code'
if GLADA_PKG_PATH not in sys.path:
    sys.path.insert(0, GLADA_PKG_PATH)

from glada.config import GLADAConfig, set_seed
from glada.models.glada_model import GLADAModel
from glada.datasets.audio import load_audio
from glada.evaluation.metrics import compute_eer, compute_eer_with_ci

set_seed(42)

MAX_LEN = 64000
EVAL_BATCH_SIZE = 48
DRIVE_BASE = '/content/drive/MyDrive/Jakob/Datasets'
LOCAL_ROOT = '/content/data'
DRIVE_EXP = '/content/drive/MyDrive/Jakob/Experiments'
os.makedirs(LOCAL_ROOT, exist_ok=True)

_eval_cfg = GLADAConfig(backbone='wavlm-base-plus')
_eval_cfg.max_audio_len = MAX_LEN

print(f'AMP: {AMP_DTYPE}')
print('Setup done.')

Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
Loaded 16 existing model results.
AMP: torch.bfloat16
Setup done.


In [2]:
# === Cell 2: Model loading + eval helpers (always run) ===

def load_glada_model(checkpoint_path, backbone='wavlm-base-plus'):
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    saved_config = ckpt.get('config', {})
    config = GLADAConfig(backbone=saved_config.get('backbone', backbone))
    for k, v in saved_config.items():
        if hasattr(config, k):
            setattr(config, k, v)
    config.__post_init__()
    model = GLADAModel(config)
    model.load_state_dict(ckpt['model'])
    model.eval()
    return model


@torch.no_grad()
def evaluate_model(model, loader, device, dataset_name, amp_dtype):
    model.eval()
    all_scores, all_labels = [], []
    for batch in tqdm(loader, desc=f'Eval {dataset_name}', leave=False):
        inp = batch['input_values'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with autocast('cuda', enabled=(device.type == 'cuda'), dtype=amp_dtype):
            out = model(inp, attention_mask=mask)
        probs = F.softmax(out['spoof_logits'], dim=1)[:, 1]
        all_scores.extend(probs.float().cpu().numpy())
        all_labels.extend(batch['task_label'].numpy())
    labels = np.array(all_labels)
    scores = np.array(all_scores)
    metrics = compute_eer_with_ci(labels, scores, n_bootstrap=1000)
    metrics['n_samples'] = len(labels)
    metrics['name'] = dataset_name
    return metrics


def _is_valid_result(ds_result):
    if not isinstance(ds_result, dict):
        return False
    if ds_result.get('eer', -1) < 0:
        return False
    if ds_result.get('n_samples', 0) == 0:
        return False
    return True


# All GLADA checkpoints to evaluate
MODELS = [
    ('full_glada',        'ablation_full_glada',           'wavlm-base-plus'),
    ('no_glada',          'ablation_no_glada',             'wavlm-base-plus'),
    ('no_vib',            'ablation_no_vib',               'wavlm-base-plus'),
    ('no_speaker',        'ablation_no_speaker',           'wavlm-base-plus'),
    ('no_cka',            'ablation_no_cka',               'wavlm-base-plus'),
    ('no_rawboost',       'ablation_no_rawboost',          'wavlm-base-plus'),
    ('no_hier',           'ablation_no_hier',              'wavlm-base-plus'),
    ('global_da_only',    'ablation_global_da_only',       'wavlm-base-plus'),
    ('hierarchical_only', 'ablation_hierarchical_only',    'wavlm-base-plus'),
    ('no_da',             'ablation_no_da',                'wavlm-base-plus'),
    ('backbone_only',     'ablation_backbone_only',        'wavlm-base-plus'),
    ('bb_wavlm-large_s42',  'ablation_full_wavlm-large',       'wavlm-large'),
    ('bb_wavlm-large_s123', 'ablation_full_wavlm-large_s123',  'wavlm-large'),
    ('bb_xlsr-300m_s42',    'ablation_full_xlsr-300m_s42',      'xlsr-300m'),
    ('bb_data2vec-large',   'ablation_full_data2vec-large',     'data2vec-large'),
    ('bb_wav2vec2-robust',  'ablation_full_wav2vec2-robust',    'wav2vec2-robust'),
]


def find_checkpoint(exp_dir):
    base = os.path.join(DRIVE_EXP, exp_dir)
    if not os.path.isdir(base):
        return None
    direct = os.path.join(base, 'best_model.pt')
    if os.path.exists(direct):
        return direct
    for sub in sorted(os.listdir(base)):
        candidate = os.path.join(base, sub, 'best_model.pt')
        if os.path.exists(candidate):
            return candidate
    return None


def eval_all_models_on_dataset(ds_name, loader):
    """Evaluate all models on a single dataset. Skip models already done."""
    n_eval = 0
    n_skip = 0
    for name, exp_dir, backbone in MODELS:
        # Check if already done
        existing = all_results.get(name, {}).get('datasets', {}).get(ds_name, {})
        if _is_valid_result(existing):
            print(f'  {name:<25s}: SKIP (EER={existing["eer"]:.2f}%)')
            n_skip += 1
            continue

        ckpt = find_checkpoint(exp_dir)
        if not ckpt:
            print(f'  {name:<25s}: CHECKPOINT NOT FOUND')
            continue

        try:
            model = load_glada_model(ckpt, backbone).to(device)
            r = evaluate_model(model, loader, device, ds_name, AMP_DTYPE)
            print(f'  {name:<25s}: EER={r["eer"]:.2f}% [{r["ci_low"]:.2f}, {r["ci_high"]:.2f}] (n={r["n_samples"]:,})')

            if name not in all_results:
                all_results[name] = {
                    'checkpoint': ckpt, 'loader_type': 'glada',
                    'backbone': backbone, 'datasets': {},
                }
            all_results[name]['datasets'][ds_name] = r
            all_results[name]['timestamp'] = datetime.datetime.now().isoformat()

            # Save after each model
            with open(RESULTS_FILE, 'w') as f:
                json.dump(all_results, f, indent=2, default=str)

            del model
            torch.cuda.empty_cache()
            n_eval += 1
        except Exception as e:
            print(f'  {name:<25s}: ERROR — {e}')

    print(f'\n{ds_name}: {n_eval} evaluated, {n_skip} skipped')


print(f'Models registered: {len(MODELS)}')

Models registered: 16


### Suppress Verbose Warnings during Evaluation

In [3]:

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='Support for mismatched key_padding_mask')
warnings.filterwarnings('ignore', message='.*torch.amp.autocast.*')
warnings.filterwarnings('ignore', message='Error loading *')
import transformers
transformers.logging.set_verbosity_error()

import logging
logging.getLogger('soundfile').setLevel(logging.ERROR)

---
## Dataset 3: ASVspoof 2021 LA
**Status:** Audio + keys both on Drive. Previous runs returned 0 samples because keys were never extracted.

**Fix:** Extract `LA-keys-full.tar.gz` which produces `keys/LA/CM/trial_metadata.txt`.

In [ ]:
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset

LA21_LOCAL = os.path.join(LOCAL_ROOT, 'asvspoof21_la')
LA21_DRIVE = os.path.join(DRIVE_BASE, 'ASVspoof2021_LA')

# Step 1: Extract audio
la21_audio_tar = None
for candidate in [os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'),
                   os.path.join(LA21_DRIVE, 'ASVspoof2021_LA_eval.tar.gz')]:
    if os.path.exists(candidate):
        la21_audio_tar = candidate
        break

la21_flac = os.path.join(LA21_LOCAL, 'ASVspoof2021_LA_eval', 'flac')
if not os.path.isdir(la21_flac):
    os.makedirs(LA21_LOCAL, exist_ok=True)
    print(f'Extracting audio from {os.path.basename(la21_audio_tar)}...')
    with tarfile.open(la21_audio_tar, 'r:gz') as tar:
        tar.extractall(path=LA21_LOCAL, filter='data')
    print('Audio extracted.')
else:
    print('Audio already extracted.')

# Step 2: Extract KEYS (the missing piece!)
la21_keys_tar = None
for candidate in [os.path.join(LA21_DRIVE, 'LA-keys-full.tar.gz'),
                   os.path.join(LA21_DRIVE, 'LA-keys-full.tar')]:
    if os.path.exists(candidate):
        la21_keys_tar = candidate
        break

la21_keys_dir = os.path.join(LA21_LOCAL, 'keys')
if not os.path.isdir(la21_keys_dir):
    if la21_keys_tar:
        print(f'Extracting keys from {os.path.basename(la21_keys_tar)}...')
        mode = 'r:gz' if la21_keys_tar.endswith('.gz') else 'r:*'
        with tarfile.open(la21_keys_tar, mode) as tar:
            tar.extractall(path=LA21_LOCAL, filter='data')
        print('Keys extracted.')
    else:
        raise FileNotFoundError('LA-keys-full.tar.gz not found! Download from asvspoof.org')
else:
    print('Keys already extracted.')

# Step 3: Find protocol file
proto_candidates = [
    os.path.join(LA21_LOCAL, 'keys', 'LA', 'CM', 'trial_metadata.txt'),
    os.path.join(LA21_LOCAL, 'ASVspoof2021_LA_eval', 'ASVspoof2021.LA.cm.eval.trl.txt'),
]
proto = None
for p in proto_candidates:
    if os.path.exists(p):
        proto = p
        break

# Step 4: Find audio directory
audio_candidates = [
    os.path.join(LA21_LOCAL, 'ASVspoof2021_LA_eval', 'flac'),
    os.path.join(LA21_LOCAL, 'flac'),
]
audio = None
for a in audio_candidates:
    if os.path.isdir(a):
        audio = a
        break

print(f'Protocol: {proto}')
print(f'Audio:    {audio}')
assert proto and audio, f'MISSING — proto={proto}, audio={audio}'

# Quick sanity: show first 3 lines of protocol
with open(proto) as f:
    for i, line in enumerate(f):
        if i < 3:
            print(f'  proto line {i}: {line.strip()}')

ds = ASVspoof2021Dataset(proto, audio, max_len=MAX_LEN)
assert len(ds) > 0, 'Dataset loaded 0 samples — check protocol format!'
loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

eval_all_models_on_dataset('2021_LA', loader)

Extracting audio from ASVspoof2021_LA_eval.tar.gz...
Audio extracted.
Extracting keys from LA-keys-full.tar.gz...
Keys extracted.
Protocol: /content/data/asvspoof21_la/keys/LA/CM/trial_metadata.txt
Audio:    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac
  proto line 0: LA_0009 LA_E_9332881 alaw ita_tx A07 spoof notrim eval
  proto line 1: LA_0009 LA_E_6866159 alaw ita_tx A07 spoof notrim eval
  proto line 2: LA_0009 LA_E_5464494 alaw sin_tx A07 spoof notrim eval
  ASVspoof 2021: 181566 samples (18452 bonafide, 163114 spoof)
  full_glada               : SKIP (EER=31.02%)
  no_glada                 : SKIP (EER=31.53%)
  no_vib                   : SKIP (EER=33.02%)
  no_speaker               : SKIP (EER=31.78%)
  no_cka                   : SKIP (EER=32.15%)
  no_rawboost              : SKIP (EER=31.98%)
  no_hier                  : SKIP (EER=33.24%)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Eval 2021_LA:   0%|          | 0/3783 [00:00<?, ?it/s]

... (33 outputs left out)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Eval 2021_LA:   0%|          | 0/3783 [00:00<?, ?it/s]

  bb_wav2vec2-robust       : EER=26.62% [26.41, 26.84] (n=181,566)

2021_LA: 9 evaluated, 7 skipped


---
## Dataset 4: ASVspoof 2021 DF
**Status:** Audio parts + keys on Drive. Needs extraction (~10-15 min for 4 part archives).

In [4]:
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset

DF_DRIVE = os.path.join(DRIVE_BASE, 'ASVspoof2021_DF')
DF_LOCAL = os.path.join(LOCAL_ROOT, 'asvspoof21_df')
os.makedirs(DF_LOCAL, exist_ok=True)

# Extract keys
df_keys_dir = os.path.join(DF_LOCAL, 'keys')
if not os.path.isdir(df_keys_dir):
    df_keys_tar = os.path.join(DF_DRIVE, 'DF-keys-full.tar')
    if os.path.exists(df_keys_tar):
        print('Extracting DF keys...')
        with tarfile.open(df_keys_tar, 'r:*') as tar:
            tar.extractall(path=DF_LOCAL)
        print('Done')
    else:
        raise FileNotFoundError(f'DF-keys-full.tar not found at {df_keys_tar}')
else:
    print('DF keys already extracted.')

# Extract audio parts — skip the _full archive (often corrupt on Drive)
df_flac = os.path.join(DF_LOCAL, 'flac')
if not os.path.isdir(df_flac):
    all_archives = sorted([f for f in os.listdir(DF_DRIVE)
                           if f.startswith('ASVspoof2021_DF_eval') and f.endswith('.tar.gz')])
    # Prefer numbered parts over _full; fall back to _full only if no parts exist
    part_archives = [f for f in all_archives if '_full' not in f]
    if not part_archives:
        part_archives = all_archives  # only _full available, try it
    print(f'Extracting {len(part_archives)} part archives (skipping _full if parts exist)...')
    for part in part_archives:
        print(f'  {part}...')
        try:
            with tarfile.open(os.path.join(DF_DRIVE, part), 'r:gz') as tar:
                tar.extractall(path=DF_LOCAL)
        except EOFError:
            print(f'    WARNING: {part} is truncated/corrupt — skipping')
            continue
    print('Audio extraction done.')
else:
    print('DF audio already extracted.')

# Find protocol + audio
proto_candidates = [
    os.path.join(DF_LOCAL, 'keys', 'DF', 'CM', 'trial_metadata.txt'),
    os.path.join(DF_LOCAL, 'trial_metadata.txt'),
]
audio_candidates = [
    os.path.join(DF_LOCAL, 'flac'),
    os.path.join(DF_LOCAL, 'ASVspoof2021_DF_eval', 'flac'),
]
proto = next((p for p in proto_candidates if os.path.exists(p)), None)
audio = next((a for a in audio_candidates if os.path.isdir(a)), None)

print(f'Protocol: {proto}')
print(f'Audio:    {audio}')
assert proto and audio, f'MISSING — proto={proto}, audio={audio}'

ds = ASVspoof2021Dataset(proto, audio, max_len=MAX_LEN)
assert len(ds) > 0, 'Dataset loaded 0 samples!'
loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Extracting DF keys...
Done
Extracting 4 part archives (skipping _full if parts exist)...
  ASVspoof2021_DF_eval_part00.tar.gz...
  ASVspoof2021_DF_eval_part01.tar.gz...
  ASVspoof2021_DF_eval_part02.tar.gz...
  ASVspoof2021_DF_eval_part03.tar.gz...
Audio extraction done.
Protocol: /content/data/asvspoof21_df/keys/DF/CM/trial_metadata.txt
Audio:    /content/data/asvspoof21_df/ASVspoof2021_DF_eval/flac
  ASVspoof 2021: 611829 samples (22617 bonafide, 589212 spoof)


In [5]:
eval_all_models_on_dataset('2021_DF', loader)

  full_glada               : SKIP (EER=27.74%)
  no_glada                 : SKIP (EER=27.96%)
  no_vib                   : SKIP (EER=29.73%)
  no_speaker               : SKIP (EER=27.57%)
  no_cka                   : SKIP (EER=29.50%)
  no_rawboost              : SKIP (EER=26.86%)
  no_hier                  : SKIP (EER=28.52%)
  global_da_only           : SKIP (EER=27.95%)
  hierarchical_only        : SKIP (EER=27.24%)
  no_da                    : SKIP (EER=27.01%)
  backbone_only            : SKIP (EER=28.50%)
  bb_wavlm-large_s42       : SKIP (EER=25.47%)
  bb_wavlm-large_s123      : SKIP (EER=25.63%)
  bb_xlsr-300m_s42         : SKIP (EER=28.17%)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.25G [00:00<?, ?B/s]

Eval 2021_DF:   0%|          | 0/12747 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Eval 2021_DF:   0%|          | 0/12747 [00:00<?, ?it/s]

  bb_wav2vec2-robust       : EER=27.84% [27.62, 28.05] (n=611,829)

2021_DF: 2 evaluated, 14 skipped


---
## Dataset 5: WaveFake
**Status:** `wavefake.tar` (22.8 GB, all 6 vocoders) on Drive. `LJSpeech-1.1.tar.bz2` (bonafide) uploaded.

**Note:** Without LJSpeech bonafide, EER cannot be computed (WaveFake only distributes spoof audio).

### Write loading and clean up scripts

In [6]:
%%writefile extract_wavefake.py
import zipfile
from tqdm import tqdm
import os

def unzip_with_progress(zip_file, target_path):
    with zipfile.ZipFile(zip_file, 'r') as zf:
        # Get total number of files for the progress bar
        total_files = len(zf.infolist())

        with tqdm(total=total_files, desc='Extracting', unit='file') as pbar:
            for member in zf.infolist():
                zf.extract(member, target_path)
                pbar.update(1)

# Example usage:
# Make sure to create the target directory if it doesn't exist
target_directory = '/content/data/wavefake/'
if not os.path.exists(target_directory):
    os.makedirs(target_directory)
unzip_with_progress('/content/data/wavefake.zip', target_directory)

Overwriting extract_wavefake.py


In [ ]:
%%writefile wavefake_bonafide_clean_up.py
#!/usr/bin/env python3
import os
import shutil
import zipfile
import tarfile
from tqdm import tqdm

# def unzip_with_progress(zip_file, target_path, desc='Extracting Zip'):
#     """Extracts a zip file with a tqdm progress bar."""
#     with zipfile.ZipFile(zip_file, 'r') as zf:
#         members = zf.infolist()
#         with tqdm(total=len(members), desc=desc, unit='file', leave=True) as pbar:
#             for member in members:
#                 zf.extract(member, target_path)
#                 pbar.update(1)

def untar_bz2_with_progress(tar_file, target_path, desc='Extracting Tar'):
    """Extracts a tar.bz2 file with a tqdm progress bar."""
    with tarfile.open(tar_file, 'r:bz2') as tf:
        members = tf.getmembers()
        with tqdm(total=len(members), desc=desc, unit='file', leave=True) as pbar:
            for member in members:
                # Note: In newer Python versions you might get a warning about filters,
                # but since we iterate and extract safely here, it works perfectly.
                tf.extract(member, path=target_path)
                pbar.update(1)

def main():
    # --- Define Paths ---
    data_dir = '/content/data'
    wavefake_zip = os.path.join(data_dir, 'wavefake.zip')
    wavefake_dir = os.path.join(data_dir, 'wavefake')

    drive_dir = '/content/drive/MyDrive/Jakob/Datasets'
    drive_wavefake_tar = os.path.join(drive_dir, 'wavefake.tar')
    drive_wavefake_zip = os.path.join(drive_dir, 'wavefake.zip')

    drive_ljspeech_tar = os.path.join(drive_dir, 'LJSpeech-1.1.tar.bz2')
    local_ljspeech_tar = os.path.join(data_dir, 'LJSpeech-1.1.tar.bz2')
    ljspeech_dir = os.path.join(wavefake_dir, 'generated_audio', 'LJSpeech-1.1')

    # --- 1 & 2. Extract Wavefake ---
    # print(">>> Step 1/3: Extracting Wavefake...")
    os.makedirs(wavefake_dir, exist_ok=True)
    # if os.path.exists(wavefake_zip):
    #     unzip_with_progress(wavefake_zip, wavefake_dir, desc='Wavefake')
    # else:
    #     print(f"  [!] {wavefake_zip} not found. Skipping extraction.")

    # --- 3. Save permanent copy to Drive ---
    print("\n>>> Step 2/3: Managing Drive files...")
    if os.path.exists(drive_wavefake_tar):
        os.remove(drive_wavefake_tar)
        print("  [-] Removed corrupted wavefake.tar")

    if os.path.exists(wavefake_zip):
        shutil.move(wavefake_zip, drive_wavefake_zip)
        print("  [+] Moved wavefake.zip to Drive")

    # --- 4. Extract bonafide to runtime (LJSpeech) ---
    print("\n>>> Step 3/3: Setting up LJSpeech...")
    if os.path.exists(drive_ljspeech_tar):
        shutil.copy2(drive_ljspeech_tar, local_ljspeech_tar)
        print("  [+] Copied LJSpeech tarball to runtime data dir")

        os.makedirs(ljspeech_dir, exist_ok=True)
        untar_bz2_with_progress(local_ljspeech_tar, ljspeech_dir, desc='LJSpeech')

        os.remove(local_ljspeech_tar)
        print("  [-] Cleaned up temporary LJSpeech tarball")
    else:
        print(f"  [!] {drive_ljspeech_tar} not found. Skipping LJSpeech extraction.")

    # --- 5. Final output count ---
    try:
        wavefake_count = len(os.listdir(wavefake_dir))
    except FileNotFoundError:
        wavefake_count = 0

    try:
        ljspeech_count = len(os.listdir(ljspeech_dir))
    except FileNotFoundError:
        ljspeech_count = 0

    print(f"\nDone — {wavefake_count} items in wavefake and {ljspeech_count} items in LJSpeech directories.")

if __name__ == '__main__':
    main()

In [7]:
## 1. Download WaveFake from Zenodo (~22 GB, uses Colab's fast network)
## Commented out = run using Colab terminal while evaluating on previous dataset
# !wget -q --show-progress -O /content/data/wavefake.zip "https://zenodo.org/records/5642694/files/generated_audio.zip?download=1"

## 2. Extract to runtime
!python3 extract_wavefake.py
!python3 wavefake_bonafide_clean_up.py
# ## 3. Save permanent copy to Drive (replaces corrupted tar)
# !rm /content/drive/MyDrive/Jakob/Datasets/wavefake.tar
# !mv /content/data/wavefake.zip /content/drive/MyDrive/Jakob/Datasets/wavefake.zip

# ## 4. Extract bonafide to runtime (LJSpeech)
# !cp /content/drive/MyDrive/Jakob/Datasets/LJSpeech-1.1.tar.bz2 /content/data/
# !mkdir /content/data/wavefake/generated_audio/LJSpeech-1.1
# !tar -xjf /content/data/LJSpeech-1.1.tar.bz2 -C /content/data/wavefake/generated_audio/LJSpeech-1.1
# !rm /content/data/LJSpeech-1.1.tar.bz2

# !echo "Done — $(ls /content/data/wavefake | wc -l) and $(ls /content/data/wavefake/generated_audio/LJSpeech-1.1 | wc -l) directories extracted"

Traceback (most recent call last):
  File "/content/extract_wavefake.py", line 20, in <module>
    unzip_with_progress('/content/data/wavefake.zip', target_directory)
  File "/content/extract_wavefake.py", line 6, in unzip_with_progress
    with zipfile.ZipFile(zip_file, 'r') as zf:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1352, in __init__
    self.fp = io.open(file, filemode)
              ^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/data/wavefake.zip'
rm: cannot remove '/content/drive/MyDrive/Jakob/Datasets/wavefake.tar': No such file or directory
mv: cannot stat '/content/data/wavefake.zip': No such file or directory
mkdir: cannot create directory ‘/content/data/wavefake/generated_audio/LJSpeech-1.1’: File exists
Done — 1 and 1 directories extracted


### Set up post extraction

In [8]:
from glada.datasets.wavefake import WaveFakeDataset

WF_LOCAL = os.path.join(LOCAL_ROOT, 'wavefake')
os.makedirs(WF_LOCAL, exist_ok=True)

# Extract WaveFake spoof audio
if not os.listdir(WF_LOCAL):
    wf_tar = os.path.join(DRIVE_BASE, 'wavefake.tar')
    wf_zip = os.path.join(DRIVE_BASE, 'wavefake.zip')
    # if os.path.exists(wf_tar):
        # print(f'Extracting wavefake.tar (~22 GB, this takes a while)...')
        # with tarfile.open(wf_tar, 'r:*') as tar:
        #     tar.extractall(path=WF_LOCAL, filter='data')
        # print('Done')
    # elif os.path.exists(wf_zip):
        # print(f'Extracting wavefake.zip...')
        # with zipfile.ZipFile(wf_zip, 'r') as zf:
        #     zf.extractall(WF_LOCAL)
        # print('Done')
    # else:
        # raise FileNotFoundError('No wavefake archive found on Drive')
else:
    print('WaveFake already extracted.')

# Extract LJSpeech bonafide into WaveFake directory
# After extraction, LJSpeech-1.1/wavs/ contains the bonafide .wav files
wf_root = WF_LOCAL
if os.path.isdir(os.path.join(WF_LOCAL, 'generated_audio')):
    wf_root = os.path.join(WF_LOCAL, 'generated_audio')

lj_dir = os.path.join(wf_root, 'LJSpeech-1.1')
if not os.path.isdir(lj_dir):
    lj_tar = os.path.join(DRIVE_BASE, 'LJSpeech-1.1.tar.bz2')
    # if os.path.exists(lj_tar):
    #     print('Extracting LJSpeech-1.1 (bonafide reference)...')
    #     with tarfile.open(lj_tar, 'r:bz2') as tar:
    #         tar.extractall(path=wf_root, filter='data')
    #     print('Done')
    # else:
    #     raise FileNotFoundError(f'LJSpeech-1.1.tar.bz2 not found at {lj_tar}')
else:
    print('LJSpeech already extracted.')

ds = WaveFakeDataset(wf_root, max_len=MAX_LEN)
n_bona = sum(1 for s in ds.samples if s['label'] == 1)
assert n_bona > 0, 'No bonafide samples! LJSpeech extraction may have failed.'
loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


WaveFake already extracted.
LJSpeech already extracted.
  WaveFake: 147366 samples (13100 bonafide, 134266 spoof)


### Run Evals

In [9]:
eval_all_models_on_dataset('WaveFake', loader)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Eval WaveFake:   0%|          | 0/3071 [00:00<?, ?it/s]

  full_glada               : EER=27.88% [27.45, 28.42] (n=147,366)


... (47 outputs left out)

  bb_data2vec-large        : EER=30.82% [30.45, 31.22] (n=147,366)


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Eval WaveFake:   0%|          | 0/3071 [00:00<?, ?it/s]

  bb_wav2vec2-robust       : EER=30.67% [30.25, 31.07] (n=147,366)

WaveFake: 16 evaluated, 0 skipped


---
## Dataset 6: CodecFake+
**Status:** `CodecFake_plus_CoSG.tar.xz` on shared drive. Needs extraction. Labels from `CoSG_labels.txt`.

**Note:** This is on a shared drive — may need different mount path. If unavailable, skip.

In [12]:

# ── Download & cache LibriSpeech test-clean as CodecFake+ bonafide ──
# CodecFake+ CoSG contains only codec-resynthesized (spoof) samples.
# Bonafide = original clean speech from LibriSpeech test-clean (~346 MB).
# Archive is cached on Drive so this only downloads once.

import os, subprocess, shutil

DRIVE_CACHE = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'
ARCHIVE_NAME = 'test-clean.tar.gz'
ARCHIVE_LOCAL = f'/content/{ARCHIVE_NAME}'
ARCHIVE_DRIVE = os.path.join(DRIVE_CACHE, ARCHIVE_NAME)
CF_LOCAL = os.path.join(LOCAL_ROOT, 'codecfake')

BONA_DIR = os.path.join(CF_LOCAL, 'bonafide')  # CodecFakeDataset sees 'bonafide' → label=1

if os.path.isdir(BONA_DIR) and len(os.listdir(BONA_DIR)) > 0:
    print(f"Bonafide already extracted at {BONA_DIR}, skipping download.")
else:
    os.makedirs(BONA_DIR, exist_ok=True)

    # 1) Download or copy archive
    if os.path.exists(ARCHIVE_DRIVE):
        print("Copying cached archive from Drive...")
        shutil.copy2(ARCHIVE_DRIVE, ARCHIVE_LOCAL)
    else:
        print("Downloading LibriSpeech test-clean from OpenSLR (~346 MB)...")
        subprocess.run([
            'wget', '-q', '--show-progress',
            'https://www.openslr.org/resources/12/test-clean.tar.gz',
            '-O', ARCHIVE_LOCAL
        ], check=True)

        # 2) Cache archive to Drive for future runs
        print("Caching archive to Drive...")
        shutil.copy2(ARCHIVE_LOCAL, ARCHIVE_DRIVE)
        print(f"Saved to {ARCHIVE_DRIVE}")

    # 3) Extract FLAC files into bonafide/ (flatten nested LibriSpeech structure)
    print("Extracting to local runtime...")
    EXTRACT_TMP = '/content/_ls_tmp'
    os.makedirs(EXTRACT_TMP, exist_ok=True)
    subprocess.run(['tar', 'xzf', ARCHIVE_LOCAL, '-C', EXTRACT_TMP], check=True)

    # LibriSpeech extracts as LibriSpeech/test-clean/speaker/chapter/*.flac
    count = 0
    for flac in Path(EXTRACT_TMP).rglob('*.flac'):
        shutil.move(str(flac), os.path.join(BONA_DIR, flac.name))
        count += 1

    # 4) Clean up
    shutil.rmtree(EXTRACT_TMP, ignore_errors=True)
    if os.path.exists(ARCHIVE_LOCAL):
        os.remove(ARCHIVE_LOCAL)

    print(f"Done: {count} bonafide FLAC files in {BONA_DIR}")

Caching archive to Drive...
Saved to /content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus/test-clean.tar.gz
Extracting to local runtime...
Done: 2620 bonafide FLAC files in /content/data/codecfake/bonafide


In [13]:
from glada.datasets.codecfake import CodecFakeDataset
import subprocess

CF_SHARED = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'
CF_LOCAL = os.path.join(LOCAL_ROOT, 'codecfake')

if not os.path.isdir(CF_SHARED):
    print(f'CodecFake+ shared drive not accessible at {CF_SHARED}')
    print('Skipping. Mount shared drive or download manually.')
else:
    # Check what we have
    print('Shared drive contents:')
    for f in os.listdir(CF_SHARED):
        print(f'  {f}')

    # Extract tar.xz to local
    cf_tar = os.path.join(CF_SHARED, 'CodecFake_plus_CoSG.tar.xz')
    if not os.path.isdir(CF_LOCAL) or not os.listdir(CF_LOCAL):
        os.makedirs(CF_LOCAL, exist_ok=True)
        if os.path.exists(cf_tar):
            print('Extracting CodecFake+ (tar.xz, may take a while)...')
            # tar.xz not natively supported by Python tarfile, use subprocess
            subprocess.run(['tar', 'xf', cf_tar, '-C', CF_LOCAL], check=True)
            print('Done')
        else:
            print(f'Archive not found: {cf_tar}')
    else:
        print('CodecFake+ already extracted.')

    if os.path.isdir(CF_LOCAL) and os.listdir(CF_LOCAL):
        # Show structure for debugging
        print('\nExtracted structure:')
        for root, dirs, files in os.walk(CF_LOCAL):
            level = root.replace(CF_LOCAL, '').count(os.sep)
            if level < 2:
                indent = '  ' * level
                print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

        ds = CodecFakeDataset(CF_LOCAL, max_len=MAX_LEN)
        n_bona = sum(1 for s in ds.samples if s['label'] == 1)
        if len(ds) > 0 and n_bona > 0:
            loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
            eval_all_models_on_dataset('CodecFake', loader)
        else:
            print(f'Cannot evaluate: {len(ds)} samples, {n_bona} bonafide')
            print('May need custom label mapping from CoSG_labels.txt')

Shared drive contents:
  CoSG_labels.txt
  CodecFake_plus_CoSG.tar.xz
  test-clean.tar.gz
CodecFake+ already extracted.

Extracted structure:
codecfake/ (0 files)
  CoSG/ (1797 files)
  bonafide/ (2620 files)
  CodecFake+: 4417 samples (2620 bonafide, 1797 spoof)


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Eval CodecFake:   0%|          | 0/93 [00:00<?, ?it/s]

  full_glada               : EER=37.77% [36.27, 39.36] (n=4,417)


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Eval CodecFake:   0%|          | 0/93 [00:00<?, ?it/s]

... (39 outputs left out)

  bb_data2vec-large        : EER=38.54% [37.02, 40.03] (n=4,417)


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Eval CodecFake:   0%|          | 0/93 [00:00<?, ?it/s]

  bb_wav2vec2-robust       : EER=39.73% [38.27, 41.30] (n=4,417)

CodecFake: 16 evaluated, 0 skipped


---
## Results Summary

In [14]:
with open(RESULTS_FILE) as f:
    all_results = json.load(f)

DS_ORDER = ['2019_LA', 'ITW', '2021_LA', '2021_DF', 'WaveFake', 'CodecFake']

header = '| Model | ' + ' | '.join(DS_ORDER) + ' |'
sep = '|---|' + '---|' * len(DS_ORDER)
print(header)
print(sep)
for name, _, _ in MODELS:
    if name not in all_results:
        continue
    row = f'| {name} |'
    ds = all_results[name].get('datasets', {})
    for d in DS_ORDER:
        r = ds.get(d, {})
        if _is_valid_result(r):
            row += f' {r["eer"]:.2f} |'
        elif isinstance(r, dict) and 'error' in r:
            row += ' ERR |'
        else:
            row += ' — |'
    print(row)

# Coverage
total = sum(1 for n in all_results for d in DS_ORDER
            if _is_valid_result(all_results[n].get('datasets', {}).get(d, {})))
possible = len(all_results) * len(DS_ORDER)
print(f'\nCoverage: {total}/{possible} cells filled')
print(f'Results saved to: {RESULTS_FILE}')

| Model | 2019_LA | ITW | 2021_LA | 2021_DF | WaveFake | CodecFake |
|---|---|---|---|---|---|---|
| full_glada | 1.04 | 19.71 | 31.02 | 27.74 | 27.88 | 37.77 |
| no_glada | 1.36 | 14.89 | 31.53 | 27.96 | 26.76 | 36.97 |
| no_vib | 0.72 | 14.58 | 33.02 | 29.73 | 19.17 | 36.88 |
| no_speaker | 0.93 | 13.48 | 31.78 | 27.57 | 27.30 | 37.58 |
| no_cka | 1.07 | 12.98 | 32.15 | 29.50 | 29.74 | 37.88 |
| no_rawboost | 1.05 | 16.66 | 31.98 | 26.86 | 23.14 | 39.28 |
| no_hier | 1.15 | 17.47 | 33.24 | 28.52 | 26.69 | 39.59 |
| global_da_only | 1.42 | 14.99 | 31.51 | 27.95 | 26.61 | 37.27 |
| hierarchical_only | 1.02 | 13.28 | 29.36 | 27.24 | 23.25 | 39.09 |
| no_da | 0.80 | 14.20 | 30.37 | 27.01 | 24.54 | 37.98 |
| backbone_only | 0.62 | 13.60 | 32.37 | 28.50 | 23.33 | 37.46 |
| bb_wavlm-large_s42 | 4.08 | 12.64 | 27.70 | 25.47 | 20.93 | 38.67 |
| bb_wavlm-large_s123 | 1.28 | 15.93 | 29.39 | 25.63 | 19.52 | 34.78 |
| bb_xlsr-300m_s42 | 0.71 | 18.40 | 32.40 | 28.17 | 23.20 | 34.18 |
| bb_data2vec

# Already Completed

---
## Dataset 1: ASVspoof 2019 LA eval
**Status:** Already on Drive. Already evaluated (Round 1). Run to verify.

In [ ]:
from glada.datasets.asvspoof import ASVspoofEvalDataset

# Extract if needed
la_tar = os.path.join(DRIVE_BASE, 'LA.tar')
la_local = os.path.join(LOCAL_ROOT, 'asvspoof')
if not os.path.isdir(os.path.join(la_local, 'LA')):
    os.makedirs(la_local, exist_ok=True)
    print('Extracting LA.tar...')
    with tarfile.open(la_tar, 'r:*') as tar:
        tar.extractall(path=la_local)
    print('Done')

proto = os.path.join(la_local, 'LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt')
audio = os.path.join(la_local, 'LA/ASVspoof2019_LA_eval/flac')
assert os.path.exists(proto) and os.path.isdir(audio), f'Missing: proto={os.path.exists(proto)}, audio={os.path.isdir(audio)}'

ds = ASVspoofEvalDataset(proto, audio, _eval_cfg)
loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'2019_LA: {len(ds)} samples')

eval_all_models_on_dataset('2019_LA', loader)

---
## Dataset 2: In-the-Wild (ITW)
**Status:** Already on Drive. Already evaluated (Round 1). Run to verify.

In [ ]:
from glada.datasets.itw import InTheWildDataset

itw_tar = os.path.join(DRIVE_BASE, 'release_in_the_wild.tar')
itw_local = os.path.join(LOCAL_ROOT, 'itw')
if not os.path.isdir(os.path.join(itw_local, 'release_in_the_wild')):
    os.makedirs(itw_local, exist_ok=True)
    print('Extracting ITW...')
    with tarfile.open(itw_tar, 'r:*') as tar:
        tar.extractall(path=itw_local)
    print('Done')

meta = os.path.join(itw_local, 'release_in_the_wild/meta.csv')
audio = os.path.join(itw_local, 'release_in_the_wild')
assert os.path.exists(meta), f'Missing meta.csv at {meta}'

ds = InTheWildDataset(meta, audio, _eval_cfg, use_labels=True)
loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'ITW: {len(ds)} samples')

eval_all_models_on_dataset('ITW', loader)